<a href="https://colab.research.google.com/github/Boran-Sert/SoftITO_Projler/blob/master/Python/30_10_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os, re, math, json, glob, pickle, logging, warnings
from dataclasses import dataclass, field, asdict
from typing import Any, Callable, Dict, List, Optional
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s", force=True)
for noisy in ("httpx", "sentence_transformers", "urllib3", "huggingface_hub", "faiss", "faiss.loader", "transformers"):
    logging.getLogger(noisy).setLevel(logging.ERROR)
log = logging.getLogger("rag")


@dataclass
class RAGConfig:
    # Parçalama
    chunk_size: int = 120            # kelime cinsinden hedef parça boyu
    chunk_overlap: int = 25          # parçalar arası örtüşme (kelime)
    add_title_to_chunk: bool = True  # embedding'e belge başlığını da ekle

    # Modeller
    embed_model: str = "BAAI/bge-m3"                 # çok dilli, Türkçe destekli
    rerank_model: str = "BAAI/bge-reranker-v2-m3"    # çok dilli cross-encoder

    # Yerel LLM (API anahtarı gerekmez)
    llm_backend: str = "auto"                     # "ollama" | "hf" | "auto"
    ollama_model: str = "Qwen/Qwen3-8B"              # alternatifler: "qwen2.5:3b", "llama3.1:8b", "gemma2:9b"
    ollama_url: str = "http://localhost:11434"
    hf_model: str = "Qwen/Qwen3-8B"  # GPU varsa: "Qwen/Qwen2.5-7B-Instruct"
    llm_temperature: float = 0.1

    # Arama
    k_dense: int = 20        # dense aramadan alınacak aday sayısı
    k_sparse: int = 20       # BM25'ten alınacak aday sayısı
    rrf_k: int = 60          # RRF sabiti
    dense_weight: float = 1.0
    sparse_weight: float = 1.0
    k_candidates: int = 12   # rerank'e girecek aday sayısı
    k_final: int = 4         # LLM'e verilecek parça sayısı
    mmr_lambda: float = 0.75 # 1 = sadece alaka, 0 = sadece çeşitlilik
    neighbor_window: int = 1 # seçilen parçanın kaç komşusu eklenecek

    # Sorgu dönüşümleri
    use_condense: bool = True     # sohbet geçmişinden bağımsız soru üret
    use_multi_query: bool = True  # sorunun farklı ifadelerini üret
    n_multi_query: int = 3
    use_hyde: bool = False        # varsayımsal cevap üretip onunla ara
    use_rerank: bool = True

cfg = RAGConfig()
cfg

RAGConfig(chunk_size=120, chunk_overlap=25, add_title_to_chunk=True, embed_model='BAAI/bge-m3', rerank_model='BAAI/bge-reranker-v2-m3', llm_backend='auto', ollama_model='Qwen/Qwen3-8B', ollama_url='http://localhost:11434', hf_model='Qwen/Qwen3-8B', llm_temperature=0.1, k_dense=20, k_sparse=20, rrf_k=60, dense_weight=1.0, sparse_weight=1.0, k_candidates=12, k_final=4, mmr_lambda=0.75, neighbor_window=1, use_condense=True, use_multi_query=True, n_multi_query=3, use_hyde=False, use_rerank=True)

In [2]:
@dataclass
class Document:
    text: str
    metadata: Dict[str, Any] = field(default_factory=dict)


def clean_text(text: str) -> str:
    text = text.replace("\r", "")
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)    # satır sonu tirelemesi: "bil-\ngi" → "bilgi"
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def load_file(path: str) -> List[Document]:
    """TXT, MD, PDF ve DOCX dosyalarını okur. PDF'te her sayfa ayrı belge olur (sayfa numarası metadata'da)."""
    ext = os.path.splitext(path)[1].lower()
    base = {"source": os.path.basename(path), "title": os.path.splitext(os.path.basename(path))[0]}
    if ext in (".txt", ".md"):
        with open(path, encoding="utf-8") as f:
            return [Document(clean_text(f.read()), base)]
    if ext == ".pdf":
        from pypdf import PdfReader
        reader = PdfReader(path)
        return [Document(clean_text(p.extract_text() or ""), {**base, "page": i + 1})
                for i, p in enumerate(reader.pages) if (p.extract_text() or "").strip()]
    if ext == ".docx":
        import docx
        d = docx.Document(path)
        return [Document(clean_text("\n\n".join(p.text for p in d.paragraphs if p.text.strip())), base)]
    raise ValueError(f"Desteklenmeyen dosya türü: {ext}")


def load_directory(path: str, exts=(".txt", ".md", ".pdf", ".docx"), extra_metadata: Optional[dict] = None) -> List[Document]:
    docs = []
    for f in sorted(glob.glob(os.path.join(path, "**", "*"), recursive=True)):
        if f.lower().endswith(exts):
            try:
                for d in load_file(f):
                    d.metadata.update(extra_metadata or {})
                    docs.append(d)
            except Exception as e:
                log.warning(f"{f} okunamadı: {e}")
    log.info(f"{len(docs)} belge yüklendi ({path})")
    return docs

In [3]:
documents = [
    Document("""Toplanan kişisel verileriniz, 6098 sayılı Türk Borçlar Kanunu, 4721 sayılı Türk Ticaret Kanunu ve ilgili yasal
yükümlülükler gereği Kanun’da öngörülen temel ilkelere uygun olarak Kanun’un 5’inci maddesinde belirtilen kişisel
veri işleme şartlarından bir veya birkaçına dayalı olarak, Kanun’un 4’üncü maddesinde belirtilen kişisel veri işleme
ilkelerine uygun şekilde işlenmektedir. Bu doğrultuda toplanan ve işlenen verileriniz ad-soy ad, T.C. kimlik numarası,
meslek, unvan, özgeçmiş, fotoğraf, video kaydı, ses kaydı, imza, iletişim bilgisi, İstanbul Ticaret Odası SoftITo Yazılım
Bilişim Akademisi’ni geliştirmeye ilişkin yükümlülüklerin yerine getirilmesi için gerekli olan bilgilerinizden
oluşmaktadır Kişisel verileriniz; İstanbul Ticaret Odası tarafından yetkilendirilmiş veri işleyen gerçek veya tüzel kişiler tarafından, eposta, telefon, internet sitesi, sosyal medya hesapları, muhtelif sözleşmeler, kağıt ortamında tutulan formlar ve
tutanaklar gibi vasıtalarla otomatik ve otomatik olmayan yöntemlerle sözlü, yazılı veya elektronik ortamda
toplanmaktadır. İstanbul Ticaret Odası tarafından işlenen kişisel veriler, İstanbul Ticaret Odası SoftITo Yazılım Bilişim
Akademisi başta olmak üzere, hukuki yükümlülüklerin yerine getirilmesi, bir hakkın tesisi ve korunması, pazarlama
faaliyetlerinin yürütülmesi, ticari ve ticari olmayan her türlü elektronik ve fiziksel iletinin gönderilmesi, her türlü
iletişim aracı ile iletişim sağlanabilmesi, anket yapılabilmesi, reklam, pazarlama ve kampanya faaliyeti yürütülmesi,
ürünler hakkında bilgi verilmesi, gerek İstanbul Ticaret Odası’nın gerek ise Sizlerin karşılıklı hak ve borçlarının
belirlenmesi, birtakım istatistiki verilere ulaşılması, katılımcıların belirlenebilmesi, beyan edilen bilgilerin teyidinin
sağlanabilmesi, proje süreçlerinin planlanması, başlatılması, icra edilmesi ve sonlandırılması, sunulan hizmet ve
yardımlardan ilgili kişilerin faydalandırılması, gerekli çalışmaların yapılması, proje, program ve hizmetlerin önerilmesi,
tanıtılması, gelecek eğitim ve projelerde içerik ve stratejilerin planlaması, İstanbul Ticaret Odası’nın ve İstanbul Ticaret
Odası ile iş ilişkisi içerisinde olan ilgili kişilerin, hukuki, teknik ve ticari-iş güvenliğinin temin edilmesi, İstanbul Ticaret
Odası’nın faaliyetlerinin yürütülmesi için gerekli olabilecek ancak temel hak ve özgürlüklerinize zarar vermemek kaydı
ile meşru menfaatleri kapsamında işlenmektedir.""",
             {"source": "softito_kvkk", "title": "İşlenen Kişisel Veriler ve Hangi Amaçla İşleneceği ", "category": "ik", "year": 2026}),

    Document("""Kişisel verileriniz; ilgili mevzuat hükümlerinin çizdiği sınırlar dahilinde, yukarıda belirtilen amaçların gerçekleştirilmesi
ve ilgili mevzuatın getirdiği yükümlülüklerin yerine getirilmesi ile sınırlı olmak üzere İstanbul Ticaret Odası tarafından,
yurt içindeki ve yurtdışındaki kanunen yetkili kamu resmi kurum ve kuruluşları, ilişkili olduğumuz üçüncü taraf gerçek
ve tüzel kişiler, hizmet sağlayıcı firmalar ve yetkilileri, iş ortakları, bankalar, iştiraklerimiz ve destek hizmeti
sağlayıcılara; hukuki yararın korunması amacıyla ilgili kamu kurum ve kuruluşlarının talepleri doğrultusunda ve talep
amaçlarıyla sınırlı olarak hukuken yetkili kamu kurum ve kuruluşlarına Kanun’un 8. ve 9. maddelerinde belirtilen kişisel
veri işleme şartları çerçevesinde aktarılabilecektir.""",
             {"source": "softito_kvkk", "title": "Kişisel Verilerin Kimlere ve Hangi Amaçla Aktarılabileceği ", "category": "ik", "year": 2026}),

    Document("""Kişisel verileriniz Kanun’un 6. maddesi kapsamında açık rızanıza dayalı olarak işlenecek ve muhafaza edilecektir. Kişisel
verilerin silinmesi, yok edilmesi, anonim hale getirilmesiyle ilgili yapılan tüm işlemler İstanbul Ticaret Odası tarafından
kayıt altına alınmaktadır. Kişisel verileriniz belirtilen amaçlarla sınırlı olmak üzere tüm kanun ve sair yasal mevzuatta
yer alan veri işleme ve zamanaşımı sürelerine riayet edilerek işlenecektir. Kanunlarda veri işleme sürelerine ilişkin
değişiklik yapılması halinde, belirlenen yeni süreler esas alınacaktır. Kişisel Kişisel verileriniz, Kanun’un madde 7/f.1.’e
göre işlenmesini gerektiren amaç ortadan kalktığında ve/veya mevzuat uyarınca verilerinizi işlememiz için zorunlu
kılındığımız zamanaşımı/saklama süreleri dolduğunda, kişisel verileriniz silinecek, yok edilecek veya anonim hale
getirilecektir.
""",
             {"source": "softito_kvkk", "title": "Kişisel Verilerin Saklanma Süresi", "category": "ik", "year": 2026}),

    Document("""Aşağıda yer alan hallerden en az birinin bulunması durumunda kişisel verilerinizin açık rızanızın bulunmadığı
durumlarda da işlenmesi mümkün olacaktır.
• Kanunlarda açıkça öngörülmesi,
• Fiili imkânsızlık nedeniyle rızasını açıklayamayacak durumda bulunan veya rızasına hukuki geçerlilik tanınmayan
kişinin kendisinin hayatı veya beden bütünlüğünün korunması için zorunlu olması,
• Bir sözleşmenin kurulması veya ifasıyla doğrudan doğruya ilgili olması kaydıyla sözleşmenin taraflarına ait
kişisel verilerin işlenmesinin gerekli olması,
• İstanbul Ticaret Odası’nın hukuki yükümlülüğünü yerine getirebilmesi için zorunlu olması,
• İlgili kişinin kendisi tarafından alenileştirilmiş olması,
• Bir hakkın tesisi, kullanılması veya korunması için veri işlemenin zorunlu olması,
• İlgili kişinin temel hak ve özgürlüklerine zarar vermemek kaydıyla, İstanbul Ticaret Odası’nın meşru menfaatleri
için veri işlenmesinin zorunlu olması.""",
             {"source": "softito_kvkk", "title": "Kanun Gereği Kişisel Veri İşlenmesinde Açık Rıza Aranmayan Haller ", "category": "ik", "year": 2026}),
]
print(len(documents), "belge")
pd.DataFrame([d.metadata for d in documents])

4 belge


,source,title,category,year
0,softito_kvkk,İşlenen Kişisel Veriler ve Hangi Amaçla İşlene...,ik,2026
1,softito_kvkk,Kişisel Verilerin Kimlere ve Hangi Amaçla Akta...,ik,2026
2,softito_kvkk,Kişisel Verilerin Saklanma Süresi,ik,2026
3,softito_kvkk,Kanun Gereği Kişisel Veri İşlenmesinde Açık Rı...,ik,2026


In [4]:
@dataclass
class Chunk:
    id: int
    text: str              # LLM'e gösterilecek metin
    embed_text: str        # vektörleştirilecek metin (başlık eklenmiş)
    doc_id: str
    position: int          # belge içindeki sıra (komşu genişletme için)
    metadata: Dict[str, Any]


_SENT_SPLIT = re.compile(r"(?<=[.!?…])\s+(?=[A-ZÇĞİÖŞÜ0-9\"“(])")

def split_sentences(text: str) -> List[str]:
    sents = []
    for para in re.split(r"\n\s*\n", text):
        para = re.sub(r"\s+", " ", para).strip()
        if para:
            sents.extend(s.strip() for s in _SENT_SPLIT.split(para) if s.strip())
    return sents


def chunk_text(text: str, size: int, overlap: int) -> List[str]:
    # Hedef boydan uzun cümleleri kelime pencerelerine böl
    units = []
    for s in split_sentences(text):
        w = s.split()
        if len(w) > size:
            step = max(1, size - overlap)
            units.extend(" ".join(w[i:i + size]) for i in range(0, len(w), step))
        else:
            units.append(s)

    chunks, cur, cur_len = [], [], 0
    for u in units:
        n = len(u.split())
        if cur and cur_len + n > size:
            chunks.append("\n".join(cur))
            # Örtüşme: son cümleleri bir sonraki parçaya taşı
            keep, kl = [], 0
            for t in reversed(cur):
                tl = len(t.split())
                if kl + tl > overlap:
                    break
                keep.insert(0, t); kl += tl
            cur, cur_len = keep, kl
        cur.append(u); cur_len += n
    if cur:
        chunks.append("\n".join(cur))
    return chunks          # parça içinde her satır bir cümle


def make_chunks(docs: List[Document], cfg: RAGConfig, start_id: int = 0) -> List[Chunk]:
    out, cid = [], start_id
    for d in docs:
        doc_id = d.metadata.get("source", "doc") + (f"#s{d.metadata['page']}" if "page" in d.metadata else "")
        title = d.metadata.get("title", "")
        for pos, t in enumerate(chunk_text(d.text, cfg.chunk_size, cfg.chunk_overlap)):
            et = f"{title}: {t}" if (cfg.add_title_to_chunk and title) else t
            out.append(Chunk(cid, t, et, doc_id, pos, dict(d.metadata)))
            cid += 1
    return out


# Demo için küçük parçalar: örnek belgeler kısa olduğu için
cfg.chunk_size, cfg.chunk_overlap = 32, 10
_demo = make_chunks(documents, cfg)
print(f"{len(_demo)} parça | ortalama {np.mean([len(c.text.split()) for c in _demo]):.0f} kelime\n")
for c in _demo[:3]:
    print(f"[{c.id}] {c.doc_id} / {c.position}: {c.text[:110].replace(chr(10), ' ')}...")

27 parça | ortalama 29 kelime

[0] softito_kvkk / 0: Toplanan kişisel verileriniz, 6098 sayılı Türk Borçlar Kanunu, 4721 sayılı Türk Ticaret Kanunu ve ilgili yasal...
[1] softito_kvkk / 1: uygun olarak Kanun’un 5’inci maddesinde belirtilen kişisel veri işleme şartlarından bir veya birkaçına dayalı ...
[2] softito_kvkk / 2: ilkelerine uygun şekilde işlenmektedir. Bu doğrultuda toplanan ve işlenen verileriniz ad-soy ad, T.C. kimlik n...


In [5]:
TR_STOP = set("""acaba ama ancak artık aslında az bana bazen bazı belki ben beni benim bir biri birkaç birçok biz bu buna
bunda bundan bunu bunun burada çok çünkü da daha de defa diye eğer en gibi hem hep hepsi her hiç için ile ise kaç kez ki kim
mi mu mü mı nasıl ne neden nerede niçin o olan olarak oldu olduğu olmak olur ona ondan onlar onu onun öyle sanki şey siz şu
tüm ve veya ya yani hangi peki var yok""".split())

def tr_lower(s: str) -> str:
    return s.replace("I", "ı").replace("İ", "i").lower()

def tokenize(text: str, stem: int = 5) -> List[str]:
    toks = re.findall(r"[0-9a-zçğıöşüâîû]+", tr_lower(text))
    return [t[:stem] if (stem and not t.isdigit()) else t for t in toks if t not in TR_STOP and len(t) > 1]


class BM25:
    def __init__(self, k1: float = 1.5, b: float = 0.75):
        self.k1, self.b = k1, b

    def fit(self, corpus_tokens: List[List[str]]):
        self.N = len(corpus_tokens)
        self.doc_len = np.array([len(d) for d in corpus_tokens], dtype=float)
        self.avgdl = self.doc_len.mean() if self.N else 1.0
        df = Counter(t for d in corpus_tokens for t in set(d))
        self.idf = {t: math.log(1 + (self.N - n + 0.5) / (n + 0.5)) for t, n in df.items()}
        self.inverted = defaultdict(list)                      # terim → [(parça, frekans)]
        for i, d in enumerate(corpus_tokens):
            for t, f in Counter(d).items():
                self.inverted[t].append((i, f))
        return self

    def scores(self, query_tokens: List[str]) -> np.ndarray:
        s = np.zeros(self.N)
        for t in set(query_tokens):
            idf = self.idf.get(t)
            if idf is None:
                continue
            for i, f in self.inverted[t]:
                s[i] += idf * f * (self.k1 + 1) / (f + self.k1 * (1 - self.b + self.b * self.doc_len[i] / self.avgdl))
        return s

print(tokenize("İzin taleplerinin İKSİS portalından yapılması zorunludur."))

['izin', 'talep', 'iksis', 'porta', 'yapıl', 'zorun']


In [6]:
class Embedder:
    def __init__(self, model_name: str):
        self.model_name = model_name
        try:
            from sentence_transformers import SentenceTransformer
            self.model = SentenceTransformer(model_name)
            self.backend = "sentence-transformers"
            self.is_e5 = "e5" in model_name.lower()
        except Exception as e:
            log.warning(f"Embedding modeli yüklenemedi ({type(e).__name__}). Yedek: karakter n-gram vektörleri.")
            from sklearn.feature_extraction.text import HashingVectorizer
            self.model = HashingVectorizer(analyzer="char_wb", ngram_range=(3, 5), n_features=2**14,
                                           alternate_sign=False, norm="l2")
            self.backend = "char-ngram (yedek)"
            self.is_e5 = False

    def _encode(self, texts: List[str], prefix: str) -> np.ndarray:
        if self.backend == "sentence-transformers":
            if self.is_e5:
                texts = [prefix + t for t in texts]
            v = self.model.encode(texts, normalize_embeddings=True, batch_size=32, show_progress_bar=False)
            return np.asarray(v, dtype="float32")
        return self.model.transform([tr_lower(t) for t in texts]).toarray().astype("float32")

    def embed_documents(self, texts: List[str]) -> np.ndarray:
        return self._encode(texts, "passage: ")

    def embed_query(self, text: str) -> np.ndarray:
        return self._encode([text], "query: ")[0]


class VectorIndex:
    """Normalize vektörlerde iç çarpım = kosinüs benzerliği."""
    def __init__(self):
        self.vectors = None
        self.faiss_index = None

    def add(self, vecs: np.ndarray):
        self.vectors = vecs if self.vectors is None else np.vstack([self.vectors, vecs])
        try:
            import faiss
            self.faiss_index = faiss.IndexFlatIP(self.vectors.shape[1])
            self.faiss_index.add(self.vectors)
        except ImportError:
            self.faiss_index = None

    def search(self, q: np.ndarray, k: int, allowed: Optional[np.ndarray] = None):
        """En benzer k parçanın (id, skor) listesi. `allowed`: filtreden geçen parça maskesi."""
        if allowed is None and self.faiss_index is not None:
            D, I = self.faiss_index.search(q[None, :].astype("float32"), min(k, len(self.vectors)))
            return [(int(i), float(d)) for i, d in zip(I[0], D[0]) if i >= 0]
        s = self.vectors @ q
        if allowed is not None:
            s = np.where(allowed, s, -np.inf)
        top = np.argsort(-s)[:k]
        return [(int(i), float(s[i])) for i in top if np.isfinite(s[i])]

In [7]:
def reciprocal_rank_fusion(ranked_lists: List[List[int]], k: int = 60, weights: Optional[List[float]] = None) -> List[tuple]:
    scores = defaultdict(float)
    weights = weights or [1.0] * len(ranked_lists)
    for w, lst in zip(weights, ranked_lists):
        for rank, idx in enumerate(lst):
            scores[idx] += w / (k + rank + 1)
    return sorted(scores.items(), key=lambda x: -x[1])


class Reranker:
    def __init__(self, model_name: str):
        try:
            from sentence_transformers import CrossEncoder
            self.model = CrossEncoder(model_name, max_length=512)
            self.available = True
        except Exception as e:
            log.warning(f"Reranker yüklenemedi ({type(e).__name__}). Reranking atlanacak, RRF sırası kullanılacak.")
            self.model, self.available = None, False

    def score(self, query: str, texts: List[str]) -> np.ndarray:
        return np.asarray(self.model.predict([(query, t) for t in texts], show_progress_bar=False), dtype=float)


def mmr_select(candidates: List[int], relevance: Dict[int, float], vectors: np.ndarray, k: int, lam: float) -> List[int]:
    if not candidates:
        return []
    r = np.array([relevance[i] for i in candidates])
    r = (r - r.min()) / (r.max() - r.min() + 1e-9)          # alaka puanlarını 0–1'e ölçekle
    rel = dict(zip(candidates, r))
    selected, pool = [], list(candidates)
    while pool and len(selected) < k:
        if not selected:
            best = max(pool, key=lambda i: rel[i])
        else:
            S = vectors[selected]
            best = max(pool, key=lambda i: lam * rel[i] - (1 - lam) * float(np.max(S @ vectors[i])))
        selected.append(best)
        pool.remove(best)
    return selected

In [8]:
class LLM:
    """API anahtarı gerektirmeyen, bilgisayarında çalışan yerel LLM.

    backend:
      "ollama" → Ollama sunucusu (önerilen: kurulumu kolay, hızlı, GPU/CPU otomatik)
      "hf"     → Hugging Face transformers; model notebook'un içinde çalışır (Colab için uygun)
      "auto"   → önce Ollama'yı dener, bulamazsa transformers'a geçer
    Hiçbiri çalışmazsa LLM adımları atlanır, cevaplar belgelerden alıntılanır.
    """
    def __init__(self, cfg: "RAGConfig"):
        self.cfg, self.backend, self.name = cfg, None, None
        order = ["ollama", "hf"] if cfg.llm_backend == "auto" else [cfg.llm_backend]
        for b in order:
            try:
                getattr(self, f"_init_{b}")()
                self.backend = b
                break
            except Exception as e:
                log.warning(f"LLM ({b}) kullanılamıyor: {type(e).__name__}: {str(e)[:150]}")
        if self.backend:
            log.info(f"LLM hazır: {self.backend} / {self.name}")
        else:
            log.warning("Yerel LLM bulunamadı: sorgu dönüşümleri atlanacak, cevaplar belgelerden alıntılanacak.")

    @property
    def available(self) -> bool:
        return self.backend is not None

    # ---------- Ollama ----------
    def _init_ollama(self):
        import requests
        r = requests.get(f"{self.cfg.ollama_url}/api/tags", timeout=3)
        r.raise_for_status()
        installed = {m["name"] for m in r.json().get("models", [])}
        want = self.cfg.ollama_model
        if want not in installed and f"{want}:latest" not in installed:
            raise RuntimeError(f"'{want}' modeli indirilmemiş. Terminalde çalıştır: ollama pull {want}  "
                               f"(yüklü modeller: {sorted(installed) or 'yok'})")
        self.name = want

    def _complete_ollama(self, messages: List[dict], max_tokens: int) -> str:
        import requests
        r = requests.post(f"{self.cfg.ollama_url}/api/chat", timeout=600, json={
            "model": self.name, "messages": messages, "stream": False,
            "options": {"temperature": self.cfg.llm_temperature, "num_predict": max_tokens, "num_ctx": 8192},
        })
        r.raise_for_status()
        return r.json()["message"]["content"]

    # ---------- Hugging Face transformers ----------
    def _init_hf(self):
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer
        name = self.cfg.hf_model
        self.tokenizer = AutoTokenizer.from_pretrained(name)
        self.model = AutoModelForCausalLM.from_pretrained(
            name, torch_dtype="auto", device_map="auto" if torch.cuda.is_available() else None)
        self.model.eval()
        self.name = name

    def _complete_hf(self, messages: List[dict], max_tokens: int) -> str:
        import torch
        text = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.tokenizer(text, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            out = self.model.generate(**inputs, max_new_tokens=max_tokens, do_sample=False,
                                      repetition_penalty=1.05, pad_token_id=self.tokenizer.eos_token_id)
        return self.tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    # ---------- Ortak arayüz ----------
    def complete(self, prompt: str, system: Optional[str] = None, max_tokens: int = 800) -> str:
        messages = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": prompt}]
        out = getattr(self, f"_complete_{self.backend}")(messages, max_tokens)
        return re.sub(r"<think>.*?</think>", "", out, flags=re.S).strip()   # "düşünen" modellerin iç notlarını at


def condense_question(llm: LLM, history: List[dict], question: str) -> str:
    if not history or not llm.available:
        return question
    h = "\n".join(f"{m['role']}: {m['content']}" for m in history[-6:])
    prompt = f"""Aşağıdaki sohbet geçmişini ve takip sorusunu kullanarak, geçmişe bakmadan tek başına anlaşılabilecek bir arama sorusu yaz.
Sadece soruyu yaz, açıklama ekleme.

Sohbet geçmişi:
{h}

Takip sorusu: {question}
Bağımsız soru:"""
    return llm.complete(prompt, max_tokens=150).strip().strip('"')


def generate_multi_queries(llm: LLM, question: str, n: int = 3) -> List[str]:
    if not llm.available:
        return []
    prompt = f"""Bir belge arama sistemi için aşağıdaki soruyu {n} farklı şekilde yeniden yaz.
Farklı eş anlamlı kelimeler ve ifadeler kullan. Her satıra bir soru yaz; numara veya açıklama ekleme.

Soru: {question}"""
    lines = [re.sub(r"^[\-\d.)\s]+", "", l).strip() for l in llm.complete(prompt, max_tokens=300).splitlines()]
    return [l for l in lines if l][:n]


def generate_hyde(llm: LLM, question: str) -> Optional[str]:
    if not llm.available:
        return None
    return llm.complete(f"Aşağıdaki soruya şirket politikası belgesinde geçebilecek kısa bir paragrafla cevap yaz:\n\n{question}",
                        max_tokens=200)

In [9]:
def matches(meta: Dict[str, Any], filters: Optional[Dict[str, Any]]) -> bool:
    if not filters:
        return True
    for key, cond in filters.items():
        val = meta.get(key)
        if callable(cond):
            if not cond(val):
                return False
        elif isinstance(cond, (list, tuple, set)):
            if val not in cond:
                return False
        elif val != cond:
            return False
    return True

In [10]:
SYSTEM_PROMPT = """Sen Örnek Teknoloji A.Ş. çalışanlarına şirket politikaları hakkında yardım eden bir asistansın.
Kurallar:
- YALNIZCA verilen bağlamdaki bilgileri kullan; kendi genel bilginle boşluk doldurma.
- Her bilginin sonunda kaynağını [1], [2] biçiminde belirt.
- Bağlamda cevap yoksa tam olarak şunu yaz: "Bu soruya belgelerde yeterli bilgi bulunamadı."
- Kaynaklar çelişiyorsa en güncel yıla ait olanı esas al ve çelişkiyi belirt.
- Kısa, net ve Türkçe cevap ver."""


class AdvancedRAG:
    def __init__(self, cfg: RAGConfig):
        self.cfg = cfg
        self.chunks: List[Chunk] = []
        self.embedder = Embedder(cfg.embed_model)
        self.reranker = Reranker(cfg.rerank_model) if cfg.use_rerank else None
        self.llm = LLM(cfg)
        self.index = VectorIndex()
        self.bm25 = BM25()
        self._by_doc: Dict[str, Dict[int, int]] = {}

    # ---------- İndeksleme ----------
    def add_documents(self, docs: List[Document]):
        new = make_chunks(docs, self.cfg, start_id=len(self.chunks))
        if not new:
            return
        self.index.add(self.embedder.embed_documents([c.embed_text for c in new]))
        self.chunks.extend(new)
        self._rebuild_sparse()
        log.info(f"{len(docs)} belge → {len(new)} parça eklendi (toplam {len(self.chunks)}) | embedding: {self.embedder.backend}")

    def _rebuild_sparse(self):
        self.bm25.fit([tokenize(c.embed_text) for c in self.chunks])
        self._by_doc = defaultdict(dict)
        for c in self.chunks:
            self._by_doc[c.doc_id][c.position] = c.id

    # ---------- Arama ----------
    def _allowed_mask(self, filters):
        if not filters:
            return None
        return np.array([matches(c.metadata, filters) for c in self.chunks])

    def _dense(self, query: str, allowed, k: int, vec: Optional[np.ndarray] = None) -> List[int]:
        q = vec if vec is not None else self.embedder.embed_query(query)
        return [i for i, _ in self.index.search(q, k, allowed)]

    def _sparse(self, query: str, allowed, k: int) -> List[int]:
        s = self.bm25.scores(tokenize(query))
        if allowed is not None:
            s = np.where(allowed, s, -np.inf)
        top = np.argsort(-s)[:k]
        return [int(i) for i in top if s[i] > 0]

    def retrieve(self, question: str, filters: Optional[dict] = None, history: Optional[List[dict]] = None,
                 mode: str = "full", k: Optional[int] = None) -> Dict[str, Any]:
        cfg, k = self.cfg, k or self.cfg.k_final
        trace = {"question": question, "mode": mode, "filters": list((filters or {}).keys())}
        allowed = self._allowed_mask(filters)

        # 1) Sorgu dönüşümleri (sadece "full" modda)
        query = question
        variants = [question]
        hyde_vec = None
        if mode == "full":
            if cfg.use_condense and history:
                query = condense_question(self.llm, history, question)
                variants = [query]
            if cfg.use_multi_query:
                variants += generate_multi_queries(self.llm, query, cfg.n_multi_query)
            if cfg.use_hyde:
                h = generate_hyde(self.llm, query)
                if h:
                    hyde_vec = self.embedder.embed_documents([h])[0]
        trace.update(standalone_query=query, query_variants=variants)

        # 2) Arama + 3) RRF
        lists, weights = [], []
        for v in variants:
            if mode in ("dense", "hybrid", "full"):
                lists.append(self._dense(v, allowed, cfg.k_dense)); weights.append(cfg.dense_weight)
            if mode in ("bm25", "hybrid", "full"):
                lists.append(self._sparse(v, allowed, cfg.k_sparse)); weights.append(cfg.sparse_weight)
        if hyde_vec is not None:
            lists.append(self._dense("", allowed, cfg.k_dense, vec=hyde_vec)); weights.append(cfg.dense_weight)

        fused = reciprocal_rank_fusion(lists, cfg.rrf_k, weights)
        if mode in ("bm25", "dense", "hybrid"):          # karşılaştırma modları: ham sıralama
            ids = [i for i, _ in fused[:k]]
            return {"hits": [self._hit(i, s) for i, s in fused[:k]], "ids": ids, "trace": trace}

        candidates = [i for i, _ in fused[:cfg.k_candidates]]
        relevance = dict(fused)

        # 4) Reranking
        if self.reranker is not None and self.reranker.available and candidates:
            rs = self.reranker.score(query, [self.chunks[i].text for i in candidates])
            relevance = dict(zip(candidates, rs.tolist()))
            candidates = sorted(candidates, key=lambda i: -relevance[i])
            trace["reranked"] = True

        # 5) MMR
        selected = mmr_select(candidates, relevance, self.index.vectors, k, cfg.mmr_lambda)
        hits = [self._hit(i, relevance[i]) for i in selected]

        # 6) Komşu genişletme
        for h in hits:
            h["context"] = self._expand(self.chunks[h["id"]], cfg.neighbor_window)
        return {"hits": hits, "ids": selected, "trace": trace}

    def _hit(self, i: int, score: float) -> Dict[str, Any]:
        c = self.chunks[i]
        return {"id": i, "score": float(score), "doc_id": c.doc_id, "text": c.text, "context": c.text, "metadata": c.metadata}

    def _expand(self, c: Chunk, window: int) -> str:
        if window <= 0:
            return c.text
        positions = self._by_doc[c.doc_id]
        seen, sents = set(), []
        for p in range(c.position - window, c.position + window + 1):
            if p in positions:
                for s in self.chunks[positions[p]].text.split("\n"):   # örtüşen cümleleri tekrar etme
                    if s not in seen:
                        seen.add(s); sents.append(s)
        return "\n".join(sents)

    # ---------- Cevap üretimi ----------
    def build_prompt(self, question: str, hits: List[dict]) -> str:
        blocks = []
        for n, h in enumerate(hits, 1):
            m = h["metadata"]
            src = f"{m.get('title', h['doc_id'])} | {m.get('source', '')}" + (f" | yıl {m['year']}" if "year" in m else "") \
                  + (f" | sayfa {m['page']}" if "page" in m else "")
            blocks.append(f"[{n}] ({src})\n{h['context']}")
        return f"### Bağlam\n" + "\n\n".join(blocks) + f"\n\n### Soru\n{question}"

    def _extractive_answer(self, question: str, hits: List[dict], n_sent: int = 3, min_coverage: float = 0.15) -> str:
        """LLM yoksa: sorunun (IDF ağırlıklı) en büyük kısmını kapsayan cümleleri kaynaklarıyla döndür.
        Not: Kelime örtüşmesine dayanır; "bilgi yok" tespiti kabadır. Güvenilir ret için LLM gerekir."""
        q = set(tokenize(question))
        max_idf = max(self.bm25.idf.values(), default=1.0)
        idf = lambda t: self.bm25.idf.get(t, max_idf)        # belgelerde hiç geçmeyen terim = en nadir terim
        total = sum(idf(t) for t in q) or 1.0
        best: Dict[str, tuple] = {}
        for n, h in enumerate(hits, 1):
            for s in h["context"].split("\n"):
                if len(s.split()) < 4 or s in best or not re.search(r"[.!?:;)]$", s):   # başlık ve tekrarları atla
                    continue
                cov = sum(idf(t) for t in q & set(tokenize(s))) / total
                if cov > 0:
                    best[s] = (cov, n)
        ranked = sorted(best.items(), key=lambda x: -x[1][0])
        if not ranked or ranked[0][1][0] < min_coverage:
            return "Bu soruya belgelerde yeterli bilgi bulunamadı."
        top = ranked[0][1][0]
        picked = [(s, n) for s, (c, n) in ranked[:n_sent] if c >= 0.5 * top]
        return "(LLM yok, belgelerden alıntı) " + " ".join(f"{s} [{n}]" for s, n in picked)

    def answer(self, question: str, filters: Optional[dict] = None, history: Optional[List[dict]] = None,
               verbose: bool = True) -> Dict[str, Any]:
        r = self.retrieve(question, filters=filters, history=history)
        hits, trace = r["hits"], r["trace"]
        q = trace.get("standalone_query", question)
        if not hits:
            ans = "Bu soruya belgelerde yeterli bilgi bulunamadı."
        elif self.llm.available:
            ans = self.llm.complete(self.build_prompt(q, hits), system=SYSTEM_PROMPT)
        else:
            ans = self._extractive_answer(q, hits)

        # Clean both answer and query variants in trace
        ans = re.sub(r'<think>.*?</think>', '', ans, flags=re.S).strip()
        if "query_variants" in trace:
            trace["query_variants"] = [re.sub(r'<think>.*?</think>', '', v, flags=re.S).strip() for v in trace["query_variants"]]

        result = {"question": question, "answer": ans, "sources": hits, "trace": trace}
        if verbose:
            show(result)
        return result


    # ---------- Kaydet / yükle ----------
    def save(self, path: str):
        os.makedirs(path, exist_ok=True)
        np.save(os.path.join(path, "vectors.npy"), self.index.vectors)
        with open(os.path.join(path, "chunks.pkl"), "wb") as f:
            pickle.dump({"chunks": [asdict(c) for c in self.chunks], "cfg": asdict(self.cfg),
                         "embed_backend": self.embedder.backend}, f)
        log.info(f"İndeks kaydedildi → {path}/")

    @classmethod
    def load(cls, path: str) -> "AdvancedRAG":
        with open(os.path.join(path, "chunks.pkl"), "rb") as f:
            data = pickle.load(f)
        rag = cls(RAGConfig(**data["cfg"]))
        if rag.embedder.backend != data["embed_backend"]:
            raise RuntimeError("Kayıtlı indeks farklı bir embedding yöntemiyle oluşturulmuş; yeniden indeksleyin.")
        rag.chunks = [Chunk(**c) for c in data["chunks"]]
        rag.index.add(np.load(os.path.join(path, "vectors.npy")))
        rag._rebuild_sparse()
        log.info(f"İndeks yüklendi: {len(rag.chunks)} parça")
        return rag


def show(result: Dict[str, Any]):
    t = result["trace"]
    print(f"❓ {result['question']}")
    if t.get("standalone_query") and t["standalone_query"] != result["question"]:
        print(f"↪ Bağımsız soru: {t['standalone_query']}")
    if len(t.get("query_variants", [])) > 1:
        print("↪ Sorgu varyantları:", " | ".join(t["query_variants"][1:]))
    print(f"\n💬 {result['answer']}\n")
    print("📚 Kaynaklar:")
    for n, h in enumerate(result["sources"], 1):
        print(f"  [{n}] {h['doc_id']:<26} skor={h['score']:+.3f}  {h['text'][:70].replace(chr(10), ' ')}...")
    print("-" * 100)

In [11]:
rag = AdvancedRAG(cfg)
rag.add_documents(documents)
print("Embedding:", rag.embedder.backend, "| Reranker:", "var" if rag.reranker and rag.reranker.available else "yok",
      "| LLM:", f"{rag.llm.backend} / {rag.llm.name}" if rag.llm.available else "yok (extractive mod)")

INFO | TensorFlow version 2.21.0 available.
INFO | JAX version 0.11.1 available.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

INFO | LLM hazır: hf / Qwen/Qwen3-8B
INFO | 4 belge → 27 parça eklendi (toplam 27) | embedding: sentence-transformers


Embedding: sentence-transformers | Reranker: var | LLM: hf / Qwen/Qwen3-8B


In [12]:
rag.add_documents(documents)

INFO | 4 belge → 27 parça eklendi (toplam 54) | embedding: sentence-transformers


In [13]:

_ = rag.answer("İşlenen Kişisel Veriler nelerdir ve Hangi Amaçla İşlenecek ?")
_ = rag.answer("Verilerim ne kadar süreyle saklanıyor ?")

❓ İşlenen Kişisel Veriler nelerdir ve Hangi Amaçla İşlenecek ?
↪ Sorgu varyantları: Hangi kişisel bilgiler işlenmektedir ve hangi amaçlarla kullanılmaktadır? | İşlenen kişisel veriler nelerdir ve bu verilerin işlenme nedenleri nedir

💬 Belgelerde işlenen kişisel verilerin türleri ve amaçlar detaylıca açıklanmamıştır. Ancak, kişisel verilerin açık rızasız işlenmesi durumları arasında **kanunlarda öngörülmesi**, **hukuki yükümlülüklerin yerine getirilmesi**, **sözleşme kurulumu veya ifasıyla doğrudan ilgili olmaları**, **hayat veya beden bütünlüğünün korunması** gibi unsurlar yer almaktadır. Bu bilgiler, veri işleme amacının **yasal yükümlülüklerin yerine getirilmesi**, **sözleşme ifası**, **hukuki hakların korunması** gibi nedenlerle olabileceğini göstermektedir.  

Kaynaklar [1]-[4]  
Bu soruya belgelerde yeterli bilgi bulunamadı.

📚 Kaynaklar:
  [1] softito_kvkk               skor=+0.889  ilkelerine uygun şekilde işlenmektedir. Bu doğrultuda toplanan ve işle...
  [2] softito_kvkk     